<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 环境与成本管控

**目标：** 通过 Colab Secrets 配置 API key，加上花费护栏，成功发起一次模型调用，并体会为什么 LLM 系统不能像普通代码那样测试。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


In [ ]:
%pip install -q groq

## 通过 Colab Secrets 配置 API key

绝不要把 API key 粘贴进 notebook 单元格。notebook 会被分享、提交到仓库、被截图，而写进单元格的 key，就是你下周要去轮换的 key。

到 **console.groq.com** 领你的免费 Groq API key（无需信用卡）。

然后把它加进 Colab：

1. 点击左侧边栏的 **钥匙图标**（"Secrets"）。
2. 点击 **Add new secret**。名称必须严格写成 `GROQ_API_KEY`，值粘贴你的 key。
3. 打开这个 notebook 的 **Notebook access** 开关。

下面的单元格会在 Colab 里读取这个 secret；如果你在本地运行，则回退到 `GROQ_API_KEY` 环境变量。


In [ ]:
import os

# 从 Colab Secrets 读取 GROQ_API_KEY（左侧边栏的钥匙图标）；如果在本地
# 运行，则从 GROQ_API_KEY 环境变量读取。
try:
    from google.colab import userdata  # 只能在 Colab 内导入
    try:
        os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    except Exception as e:
        # 在 Colab 里，但 secret 缺失或 notebook access 未打开。
        raise RuntimeError(
            "Couldn't read GROQ_API_KEY from Colab Secrets. Click the key icon "
            "in the left sidebar, add a secret named exactly GROQ_API_KEY, and "
            "flip 'Notebook access' on for this notebook, then re-run this cell."
        ) from e
except ImportError:
    # 不在 Colab 里——从环境变量取 key。
    if not os.environ.get('GROQ_API_KEY'):
        raise RuntimeError(
            'GROQ_API_KEY is not set. Run `export GROQ_API_KEY=...` in your '
            'shell before launching Jupyter, or set it in os.environ.'
        )

from groq import Groq

# max_retries=8（SDK 默认是 2）：免费档的每分钟 token 上限很低，所以一
# 串连续调用可能撞上 HTTP 429。SDK 会对 429 自动退避重试，并遵循服务端
# 的 retry-after 提示，所以把上限调高，只是把这些短暂的等待窗口透明地等过去，
# 而不是报错。没超限时它不产生任何成本。
client = Groq(max_retries=8)
MODEL = 'openai/gpt-oss-120b'  # 如果遇到 404 就更新它——运行下面的单元格看看有哪些可用
print('Groq client ready. Default MODEL:', MODEL)

## 查看可用模型

Groq 的模型目录会随时间变化：有新模型加入，有模型改名，也有模型下线。所以不要把模型名写死然后忘了它。如果调用报 `404 / model_not_found`，运行下面的单元格看看当前有哪些可用，然后更新上面的 `MODEL`。

这个目录里也包含非对话模型（语音转文字、文字转语音、安全/护栏模型）。下面的单元格会把它们过滤掉，让你只在对话模型里挑。

**怎么选：**
- **默认选最大的对话模型**（名字里参数量最高的，所以 `120b` 优于 `20b`）。在你迭代的阶段，它强到足以应付几乎所有任务。
- **批量、重复性的活儿换小模型。** 分类、对成百上千行做抽取、评测循环里的 judge 调用，以及任何要跑 N 次、速度与成本都会成倍放大的场景。

In [ ]:
# 列出你 Groq 账号当前可用的对话模型。
# 如果上面的 MODEL 报 404，从这个列表里挑一个替换它。

# 标记非对话模型（语音转文字、文字转语音、安全模型）的子串。
NON_CHAT = ('whisper', 'tts', 'guard', 'embed', 'distil-whisper')

models = client.models.list()
chat_models = sorted(
    m.id for m in models.data
    if not any(tag in m.id.lower() for tag in NON_CHAT)
)

print(f'{len(chat_models)} chat models available:\n')
for m in chat_models:
    marker = '  <- current MODEL' if m == MODEL else ''
    print(f'  {m}{marker}')

if MODEL not in chat_models:
    print(f'\n⚠️  Current MODEL ({MODEL!r}) is not in the list above — pick one and update the MODEL cell.')

## 第一次调用

一次请求，一次响应。Groq 遵循 OpenAI 兼容格式：`response.choices[0].message.content` 是文本，`response.usage` 是这次调用的计费记录。


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[{'role': 'user', 'content': 'In two sentences: what does a forward-deployed engineer do?'}],
)

print(response.choices[0].message.content)
print()
print('finish_reason:', response.choices[0].finish_reason)
print('usage:', response.usage)


## 冲击：这不是普通软件

在学其他任何东西之前，先体会一件事——它让你在 LLM 上构建系统的方式，和你交付过的所有后端系统都不同。运行下面的单元格（**同一个请求，跑三次**），看着答案彼此*对不上*。然后看着一个严格的输出契约间歇性失效。

在普通软件里，同样的输入必然得到同样的输出，一个测试今天通过就永远通过。而在这里，「我试的时候是好的」对下一次调用毫无保证。就这一个事实，决定了整门课为什么围绕**度量**行为来组织，而不是靠肉眼看；也决定了为什么第 02 节要在你构建任何需要信任的东西之前，先把评测装进去。

In [ ]:
# 同一个 prompt，跑三次。在普通代码里，这本该得到三个完全相同的结果。
prompt = 'Name one benefit of RAG. Answer in exactly 3 words.'
print('--- same prompt x3 (temperature=1) ---')
for i in range(3):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=20, temperature=1,
        messages=[{'role': 'user', 'content': prompt}],
    )
    print(f'  {i+1}: {r.choices[0].message.content.strip()!r}')

# 现在换一个严格的输出*契约*："只回一个数字"。多跑几次，看它
# 偶尔加上额外的词、标点或一整句话——这种漂移在生产环境里会
# 悄无声息地弄坏下游的 `int(response)`。
print('\n--- strict contract: "reply with ONLY the number" x3 ---')
for i in range(3):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=20, temperature=1,
        messages=[{'role': 'user',
                   'content': 'How many bytes in a kilobyte? Reply with ONLY the number, nothing else.'}],
    )
    out = r.choices[0].message.content.strip()
    parses = out.isdigit()
    print(f'  {i+1}: {out!r}   int()-parseable? {parses}')

## 读懂 `response.usage`

每个响应都带有精确的 token 记账。这就是你判断成本的依据：

- `prompt_tokens`：输入 token（你发送的一切：system、历史记录、这条消息）
- `completion_tokens`：输出 token（模型生成的内容）
- `total_tokens`：两者之和

Groq 没有 prompt 缓存，所以不存在需要计算的缓存字段。成本就是每个桶的 token 数乘以各自的单价，下面这个花费护栏需要做的也只有这些。

In [ ]:
import time  # 供末尾的练习使用

PRICES = {
    # 每百万 token 的美元价：(输入, 输出)。Groq 免费档不花钱——
    # 这些价格是为了让对比练习里的成本计算真实，也为你日后
    # 迁移到付费档做准备。随时到 console.groq.com/pricing 核对当前价格。
    'openai/gpt-oss-120b': (0.90, 0.90),
    'openai/gpt-oss-20b':  (0.15, 0.15),
}
DEFAULT_RATE = (0.90, 0.90)  # 不在 PRICES 里的模型用这个兜底


class BudgetExceeded(Exception):
    """当某次调用会让会话花费超出预算时抛出。"""


class SpendGuard:
    """对 client.chat.completions.create 的开箱即用包装。
    记录每次调用的估算成本，维护会话累计花费，并在预算耗尽后拒绝发起调用。签名与被包装的
    方法一致，所以你可以在任何地方把 `client.chat.completions.create(...)` 换成
    `guard.create(...)`。
    """

    def __init__(self, budget_usd=0.50):
        self.budget = budget_usd
        self.spent = 0.0
        self.calls = 0

    def _cost(self, model, usage):
        inp_rate, out_rate = PRICES.get(model, DEFAULT_RATE)
        return (usage.prompt_tokens * inp_rate + usage.completion_tokens * out_rate) / 1_000_000

    def create(self, **kwargs):
        if self.spent >= self.budget:
            raise BudgetExceeded(
                f'Session spend ${self.spent:.4f} >= budget ${self.budget:.2f} — refusing to call.'
            )
        response = client.chat.completions.create(**kwargs)
        cost = self._cost(kwargs['model'], response.usage)
        self.spent += cost
        self.calls += 1
        print(
            f'[spend_guard] call {self.calls}: '
            f'{response.usage.prompt_tokens} in / {response.usage.completion_tokens} out '
            f'~= ${cost:.5f}  (session: ${self.spent:.5f} / ${self.budget:.2f})'
        )
        return response


guard = SpendGuard(budget_usd=0.25)
print(f'SpendGuard ready — budget ${guard.budget:.2f}.')

In [ ]:
# 正常用法：签名与 client.chat.completions.create 相同，另外多一个累计记账。
resp = guard.create(
    model=MODEL,
    max_tokens=100,
    messages=[{'role': 'user', 'content': 'One sentence: why do teams add spend guards around LLM calls?'}],
)
print(resp.choices[0].message.content)


In [ ]:
# 再看失败路径：故意给一个极小的预算，第一次调用之后就会被拦下。
tiny = SpendGuard(budget_usd=0.0001)
try:
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi.'}])  # 这一次会成功……
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi again.'}])  # ……这一次会抛异常
except BudgetExceeded as e:
    print('Guard tripped as expected:', e)


## 选模型（在 Groq 上）

名字会变，原则不会。Groq 免费提供两个主要档次：

- **120B 档**（`openai/gpt-oss-120b`）：迭代时的默认选择。强到足以应付几乎所有任务。
- **20B 档**（`openai/gpt-oss-20b`）：批量和廉价操作。分类、对上万行做抽取、评测循环里的 judge 调用，以及任何要乘以 N 的场景。

工作原则是：**用便宜的开发，用目标模型评测。** 用 20B 模型把 pipeline 搭起来，然后在提交之前，拿你的评测集跑一遍真正要上线的 120B 模型。prompt 的行为会在档次之间发生偏移（一个在 20B 上调好的 prompt 在 120B 上可能表现不同），所以「用目标模型评测」这一步不能省。

In [ ]:
# 同一个 prompt 跑两个档次——比较成本和体感。（2 次 API 调用。）
prompt = 'Classify the sentiment of this review as positive, negative, or mixed: '\
         '"The battery life is great but the screen scratched in a week." Reply with one word.'

for model in ['openai/gpt-oss-20b', 'openai/gpt-oss-120b']:
    r = guard.create(model=model, max_tokens=10,
                     messages=[{'role': 'user', 'content': prompt}])
    print(f'{model}: {r.choices[0].message.content.strip()}')

跑一下，注意两个档次在这个简单任务上都做对了，而这正是要点所在。如果小模型能通过某个任务的评测，那么把它路由到大模型纯粹是在烧利润。

## 练习

1. 扩展 `SpendGuard.create`，记录每次调用的实际耗时（用 `time.monotonic()` 包住请求），并加一个 `.report()` 方法，打印总成本、总调用次数以及 p50/最大延迟。
2. 拿一个你在意的 prompt，用两种方式表述：简洁的项目符号风格，以及完整的散文风格。比较两者的 `prompt_tokens`（两个都用 `max_tokens=1` 发送以控制成本），并算出在每月 10 万次调用下的成本差异。
3. 给 `SpendGuard` 加一个 `hard_max_tokens` 选项，拒绝任何请求输出 token 超过 N 的调用，这样像 `max_tokens=100000` 这样的笔误就不会溜过去。
4. Groq 免费档有速率限制（每分钟请求数和每分钟 token 数）。给 `SpendGuard.create` 加一个重试循环，捕获 `groq.RateLimitError` 并用 `time.sleep` 退避。